<a href="https://colab.research.google.com/github/pandeydevangshu12/AI_TRAFFIC_POLICE/blob/main/Notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Untitled document (3).gdoc', 'JEE MAIN REGISTRATION.pdf', 'Appraisal For Confirmation Format - Revised (1).doc', 'Appraisal For Confirmation Format - Revised.doc', 'schedule.xlsx', 'JEE MAINS 24 TH JULY 2024 SHIFT-2.pdf', 'STATUS.xlsx', 'DEVANGSHU ALL DOCUMENTS SOFT COPY', 'Figurine details A 1 7 scale figurine of a charac....gdoc', 'Google Earth', 'give me this in a structred manner.gsheet', 'Earth\nfunctions as a life support system by provi....gsheet', 'Copy of WAT-Test.pdf', 'Colab Notebooks', 'can you like mark me for this out of 10.gsheet', 'Untitled document (2).gdoc', 'SSB_SOFT_COPIES', 'PlantGuard_AI_Project_Report_Updated (1).docx', 'PlantGuard_AI_Project_Report_Updated.docx', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'video_20260521_120652.mp4.adding', 'Devangshu_Pandey_CV.pdf', 'Google AI Studio', 'AI_TRAFFIC_POLICE']


In [3]:
import os

PROJECT = "/content/drive/MyDrive/AI_TRAFFIC_POLICE"

print("Project exists:", os.path.exists(PROJECT))
print("\nProject contents:")
print(os.listdir(PROJECT))

Project exists: True

Project contents:
['README.md', '.gitignore', 'test', 'valid', 'src', 'train', 'docs', '.vscode', 'outputs', '.git']


In [4]:
from pathlib import Path

ROOT = Path(PROJECT)

for split in ["train", "valid", "test"]:
    split_dir = ROOT / split

    images = list(split_dir.glob("*.jpg"))
    csv_exists = (split_dir / "_classes.csv").exists()

    print(
        f"{split:6} | "
        f"Images: {len(images):,} | "
        f"CSV: {csv_exists}"
    )

train  | Images: 13,300 | CSV: True
valid  | Images: 3,798 | CSV: True
test   | Images: 1,900 | CSV: True


In [5]:
!cp -r "/content/drive/MyDrive/AI_TRAFFIC_POLICE" "/content/"


In [6]:
from pathlib import Path
import sys
import os

PROJECT = Path("/content/AI_TRAFFIC_POLICE")

os.chdir(PROJECT)

if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))

print("Working directory:", os.getcwd())
print("Project exists:", PROJECT.exists())

Working directory: /content/AI_TRAFFIC_POLICE
Project exists: True


In [7]:
import torch

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory / 1024**3,
            2
        ),
        "GB"
    )

Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [10]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Current directory:", Path.cwd())

import dataset

print("dataset.py loaded from:")
print(dataset.__file__)

print("\nROOT used by dataset.py:")
print(dataset.ROOT)

print("\nROOT exists:")
print(dataset.ROOT.exists())

print("\nROOT contents:")
print(list(dataset.ROOT.iterdir())[:10])

Python: /usr/bin/python3
Current directory: /content/AI_TRAFFIC_POLICE
dataset.py loaded from:
/content/AI_TRAFFIC_POLICE/src/dataset.py

ROOT used by dataset.py:
/content/AI_TRAFFIC_POLICE

ROOT exists:
True

ROOT contents:
[PosixPath('/content/AI_TRAFFIC_POLICE/README.md'), PosixPath('/content/AI_TRAFFIC_POLICE/.git'), PosixPath('/content/AI_TRAFFIC_POLICE/.gitignore'), PosixPath('/content/AI_TRAFFIC_POLICE/src'), PosixPath('/content/AI_TRAFFIC_POLICE/outputs'), PosixPath('/content/AI_TRAFFIC_POLICE/train'), PosixPath('/content/AI_TRAFFIC_POLICE/valid'), PosixPath('/content/AI_TRAFFIC_POLICE/.vscode'), PosixPath('/content/AI_TRAFFIC_POLICE/test'), PosixPath('/content/AI_TRAFFIC_POLICE/docs')]


In [11]:
from pathlib import Path

ROOT = Path("/content/AI_TRAFFIC_POLICE")

train_dir = ROOT / "train"

print("Train exists:", train_dir.exists())
print("CSV exists:", (train_dir / "_classes.csv").exists())

print("\nNumber of JPGs:")
print(len(list(train_dir.glob("*.jpg"))))

print("\nCSV:")
print((train_dir / "_classes.csv").read_text()[:200])

Train exists: True
CSV exists: True

Number of JPGs:
13300

CSV:
filename, bus, car, motorcycle, truck
000000347908_jpg.rf.90f993384ab1d36e92a1e10041fc4dad.jpg, 0, 1, 0, 0
000000470364_jpg.rf.90fc148184f4f557f4bb0dc3a64d0350.jpg, 0, 1, 0, 1
000000085589_jpg.rf.90fe


In [13]:
from dataset import create_dataloaders

train_loader, valid_loader, test_loader = create_dataloaders(
    batch_size=64,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches:", len(test_loader))

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

Train batches: 208
Valid batches: 60
Test batches: 30
Images: torch.Size([64, 3, 224, 224])
Labels: torch.Size([64, 4])
Images dtype: torch.float32
Labels dtype: torch.float32


In [15]:
from model import VehicleCNN

print("VehicleCNN imported successfully")
images = images.cuda()
labels = labels.cuda()

model = VehicleCNN(num_classes=4).cuda()

with torch.no_grad():
    output = model(images)

print(output.shape)
print(output.device)

VehicleCNN imported successfully
torch.Size([64, 4])
cuda:0


In [16]:
import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

DEVICE = torch.device("cuda")

CLASS_NAMES = [
    "bus",
    "car",
    "motorcycle",
    "truck",
]

EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [17]:
model = VehicleCNN(
    num_classes=4
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(model)
print("\nParameters:", sum(p.numel() for p in model.parameters()))

VehicleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, a

In [18]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, labels in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        batch_size = images.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    return total_loss / total_samples

In [19]:
def evaluate(
    model,
    loader,
    criterion,
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            probabilities = torch.sigmoid(
                logits
            )

            batch_size = images.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            all_labels.append(
                labels.cpu().numpy()
            )

            all_probs.append(
                probabilities.cpu().numpy()
            )

    all_labels = np.concatenate(
        all_labels
    )

    all_probs = np.concatenate(
        all_probs
    )

    predictions = (
        all_probs >= THRESHOLD
    ).astype(int)

    precision = precision_score(
        all_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    recall = recall_score(
        all_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    per_class_f1 = f1_score(
        all_labels,
        predictions,
        average=None,
        zero_division=0,
    )

    return (
        total_loss / total_samples,
        precision,
        recall,
        f1,
        per_class_f1,
    )

In [20]:
best_val_f1 = -1

history = {
    "train_loss": [],
    "val_loss": [],
    "precision": [],
    "recall": [],
    "f1": [],
}

BEST_MODEL = (
    Path("/content/AI_TRAFFIC_POLICE")
    / "outputs"
    / "training"
    / "best_model.pth"
)

BEST_MODEL.parent.mkdir(
    parents=True,
    exist_ok=True
)

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
    )

    (
        val_loss,
        precision,
        recall,
        f1,
        per_class_f1,
    ) = evaluate(
        model,
        valid_loader,
        criterion,
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["precision"].append(
        precision
    )

    history["recall"].append(
        recall
    )

    history["f1"].append(
        f1
    )

    print(
        f"\nEpoch {epoch:02d}/{EPOCHS}"
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Precision  : {precision:.4f}"
    )

    print(
        f"Recall     : {recall:.4f}"
    )

    print(
        f"F1         : {f1:.4f}"
    )

    print("Per-class F1:")

    for name, score in zip(
        CLASS_NAMES,
        per_class_f1
    ):
        print(
            f"  {name:12}: {score:.4f}"
        )

    if f1 > best_val_f1:

        best_val_f1 = f1

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "epoch": epoch,

                "val_f1": f1,

                "class_names":
                    CLASS_NAMES,

                "threshold":
                    THRESHOLD,
            },
            BEST_MODEL,
        )

        print("✓ Best model saved")

print("\nTraining complete!")
print("Best validation F1:", best_val_f1)
print("Model:", BEST_MODEL)


Epoch 01/15
Train Loss : 0.5574
Val Loss   : 0.5562
Precision  : 0.4866
Recall     : 0.3365
F1         : 0.3564
Per-class F1:
  bus         : 0.2363
  car         : 0.7684
  motorcycle  : 0.4101
  truck       : 0.0107
✓ Best model saved

Epoch 02/15
Train Loss : 0.5335
Val Loss   : 0.6619
Precision  : 0.2538
Recall     : 0.4288
F1         : 0.3119
Per-class F1:
  bus         : 0.0000
  car         : 0.7978
  motorcycle  : 0.4497
  truck       : 0.0000

Epoch 03/15
Train Loss : 0.5269
Val Loss   : 0.5660
Precision  : 0.6057
Recall     : 0.4360
F1         : 0.4158
Per-class F1:
  bus         : 0.4873
  car         : 0.7498
  motorcycle  : 0.3795
  truck       : 0.0466
✓ Best model saved

Epoch 04/15
Train Loss : 0.5226
Val Loss   : 0.5480
Precision  : 0.7240
Recall     : 0.3378
F1         : 0.3235
Per-class F1:
  bus         : 0.0073
  car         : 0.8101
  motorcycle  : 0.4749
  truck       : 0.0016

Epoch 05/15
Train Loss : 0.5190
Val Loss   : 0.5360
Precision  : 0.7840
Recall     : 

In [21]:
import torch
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

DEVICE = torch.device("cuda")

BEST_MODEL = "/content/AI_TRAFFIC_POLICE/outputs/training/best_model.pth"

checkpoint = torch.load(
    BEST_MODEL,
    map_location=DEVICE
)

model = VehicleCNN(num_classes=4).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded best model")
print("Saved epoch:", checkpoint["epoch"])
print("Saved validation F1:", checkpoint["val_f1"])

Loaded best model
Saved epoch: 6
Saved validation F1: 0.5172046431263455


In [22]:
all_labels = []
all_probabilities = []

model.eval()

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        probabilities = torch.sigmoid(
            logits
        )

        all_labels.append(
            labels.numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

all_labels = np.concatenate(
    all_labels,
    axis=0
)

all_probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

print("Labels shape:",
      all_labels.shape)

print("Probabilities shape:",
      all_probabilities.shape)

Labels shape: (3798, 4)
Probabilities shape: (3798, 4)


In [23]:
CLASS_NAMES = [
    "bus",
    "car",
    "motorcycle",
    "truck",
]

thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

best_thresholds = {}
best_f1_scores = {}

for class_index, class_name in enumerate(CLASS_NAMES):

    y_true = all_labels[:, class_index]
    y_prob = all_probabilities[:, class_index]

    best_f1 = 0.0
    best_threshold = 0.5

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold

    best_thresholds[class_name] = best_threshold
    best_f1_scores[class_name] = best_f1

    print(
        f"{class_name:12} | "
        f"Best threshold: {best_threshold:.2f} | "
        f"F1: {best_f1:.4f}"
    )

bus          | Best threshold: 0.45 | F1: 0.5311
car          | Best threshold: 0.40 | F1: 0.8106
motorcycle   | Best threshold: 0.25 | F1: 0.5693
truck        | Best threshold: 0.35 | F1: 0.5273


In [24]:
old_thresholds = {
    "bus": 0.5,
    "car": 0.5,
    "motorcycle": 0.5,
    "truck": 0.5,
}

old_predictions = np.zeros_like(
    all_labels
)

new_predictions = np.zeros_like(
    all_labels
)

for i, class_name in enumerate(CLASS_NAMES):

    old_predictions[:, i] = (
        all_probabilities[:, i]
        >= old_thresholds[class_name]
    )

    new_predictions[:, i] = (
        all_probabilities[:, i]
        >= best_thresholds[class_name]
    )

In [25]:
print("=" * 60)
print("THRESHOLD COMPARISON")
print("=" * 60)

for i, class_name in enumerate(CLASS_NAMES):

    old_f1 = f1_score(
        all_labels[:, i],
        old_predictions[:, i],
        zero_division=0
    )

    new_f1 = f1_score(
        all_labels[:, i],
        new_predictions[:, i],
        zero_division=0
    )

    old_precision = precision_score(
        all_labels[:, i],
        old_predictions[:, i],
        zero_division=0
    )

    new_precision = precision_score(
        all_labels[:, i],
        new_predictions[:, i],
        zero_division=0
    )

    old_recall = recall_score(
        all_labels[:, i],
        old_predictions[:, i],
        zero_division=0
    )

    new_recall = recall_score(
        all_labels[:, i],
        new_predictions[:, i],
        zero_division=0
    )

    print(f"\n{class_name.upper()}")

    print(
        f"Threshold: "
        f"{old_thresholds[class_name]:.2f}"
        f" → "
        f"{best_thresholds[class_name]:.2f}"
    )

    print(
        f"F1: "
        f"{old_f1:.4f}"
        f" → "
        f"{new_f1:.4f}"
    )

    print(
        f"Precision: "
        f"{old_precision:.4f}"
        f" → "
        f"{new_precision:.4f}"
    )

    print(
        f"Recall: "
        f"{old_recall:.4f}"
        f" → "
        f"{new_recall:.4f}"
    )

THRESHOLD COMPARISON

BUS
Threshold: 0.50 → 0.45
F1: 0.5191 → 0.5311
Precision: 0.5157 → 0.4830
Recall: 0.5226 → 0.5900

CAR
Threshold: 0.50 → 0.40
F1: 0.8068 → 0.8106
Precision: 0.7082 → 0.6855
Recall: 0.9373 → 0.9914

MOTORCYCLE
Threshold: 0.50 → 0.25
F1: 0.3884 → 0.5693
Precision: 0.8916 → 0.6120
Recall: 0.2483 → 0.5322

TRUCK
Threshold: 0.50 → 0.35
F1: 0.3545 → 0.5273
Precision: 0.4965 → 0.3823
Recall: 0.2756 → 0.8496


In [26]:
old_macro_f1 = f1_score(
    all_labels,
    old_predictions,
    average="macro",
    zero_division=0
)

new_macro_f1 = f1_score(
    all_labels,
    new_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("OVERALL RESULT")
print("=" * 60)

print(
    f"Original macro F1 : "
    f"{old_macro_f1:.4f}"
)

print(
    f"Optimized macro F1: "
    f"{new_macro_f1:.4f}"
)

print(
    f"Improvement        : "
    f"{new_macro_f1 - old_macro_f1:+.4f}"
)


OVERALL RESULT
Original macro F1 : 0.5172
Optimized macro F1: 0.6096
Improvement        : +0.0924


In [28]:
import pandas as pd
import torch

TRAIN_CSV = "/content/AI_TRAFFIC_POLICE/train/_classes.csv"

df = pd.read_csv(TRAIN_CSV)

# Remove whitespace from column names
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns.tolist())

CLASS_NAMES = [
    "bus",
    "car",
    "motorcycle",
    "truck",
]

pos_counts = df[CLASS_NAMES].sum()
neg_counts = len(df) - pos_counts

pos_weight = neg_counts / pos_counts

print("\nPositive counts:")
print(pos_counts)

print("\nNegative counts:")
print(neg_counts)

print("\nPositive weights:")
print(pos_weight)

Columns:
['filename', 'bus', 'car', 'motorcycle', 'truck']

Positive counts:
bus           2915
car           8976
motorcycle    2593
truck         4455
dtype: int64

Negative counts:
bus           10385
car            4324
motorcycle    10707
truck          8845
dtype: int64

Positive weights:
bus           3.562607
car           0.481729
motorcycle    4.129194
truck         1.985410
dtype: float64


In [29]:
pos_weight_tensor = torch.tensor(
    [3.562607, 0.481729, 4.129194, 1.985410],
    dtype=torch.float32,
    device=DEVICE,
)

criterion_weighted = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

print("Positive weights:", pos_weight_tensor)
print("Loss function:", criterion_weighted)

Positive weights: tensor([3.5626, 0.4817, 4.1292, 1.9854], device='cuda:0')
Loss function: BCEWithLogitsLoss()


In [30]:
weighted_model = VehicleCNN(
    num_classes=4
).to(DEVICE)

weighted_optimizer = torch.optim.Adam(
    weighted_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print(
    "Parameters:",
    sum(p.numel() for p in weighted_model.parameters())
)

Parameters: 390404


In [31]:
WEIGHTED_EPOCHS = 15

weighted_best_f1 = -1.0

WEIGHTED_MODEL_PATH = (
    "/content/AI_TRAFFIC_POLICE/"
    "outputs/training/best_weighted_model.pth"
)

weighted_history = []

for epoch in range(1, WEIGHTED_EPOCHS + 1):

    train_loss = train_one_epoch(
        weighted_model,
        train_loader,
        criterion_weighted,
        weighted_optimizer,
    )

    (
        val_loss,
        precision,
        recall,
        f1,
        per_class_f1,
    ) = evaluate(
        weighted_model,
        valid_loader,
        criterion_weighted,
    )

    weighted_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

    print(f"\nEpoch {epoch:02d}/{WEIGHTED_EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1         : {f1:.4f}")

    print("Per-class F1:")

    for name, score in zip(
        CLASS_NAMES,
        per_class_f1
    ):
        print(
            f"  {name:12}: {score:.4f}"
        )

    if f1 > weighted_best_f1:

        weighted_best_f1 = f1

        torch.save(
            {
                "model_state_dict":
                    weighted_model.state_dict(),

                "epoch": epoch,

                "val_f1": f1,

                "class_names":
                    CLASS_NAMES,
            },
            WEIGHTED_MODEL_PATH,
        )

        print("✓ Best weighted model saved")

print("\n" + "=" * 60)
print("WEIGHTED TRAINING COMPLETE")
print("=" * 60)

print(
    f"Best weighted validation F1: "
    f"{weighted_best_f1:.4f}"
)

print(
    f"Model saved at:\n"
    f"{WEIGHTED_MODEL_PATH}"
)


Epoch 01/15
Train Loss : 0.8378
Val Loss   : 0.8260
Precision  : 0.4834
Recall     : 0.5190
F1         : 0.4655
Per-class F1:
  bus         : 0.3732
  car         : 0.5575
  motorcycle  : 0.4743
  truck       : 0.4569
✓ Best weighted model saved

Epoch 02/15
Train Loss : 0.8028
Val Loss   : 0.9974
Precision  : 0.4547
Recall     : 0.4955
F1         : 0.3653
Per-class F1:
  bus         : 0.4491
  car         : 0.4852
  motorcycle  : 0.3627
  truck       : 0.1641

Epoch 03/15
Train Loss : 0.7911
Val Loss   : 0.9240
Precision  : 0.5013
Recall     : 0.5745
F1         : 0.4578
Per-class F1:
  bus         : 0.3892
  car         : 0.4287
  motorcycle  : 0.5374
  truck       : 0.4758

Epoch 04/15
Train Loss : 0.7826
Val Loss   : 0.9339
Precision  : 0.5652
Recall     : 0.5648
F1         : 0.4586
Per-class F1:
  bus         : 0.4536
  car         : 0.8004
  motorcycle  : 0.1605
  truck       : 0.4199

Epoch 05/15
Train Loss : 0.7766
Val Loss   : 0.7663
Precision  : 0.4816
Recall     : 0.7464
F1 

In [32]:
checkpoint = torch.load(
    "/content/AI_TRAFFIC_POLICE/outputs/training/best_weighted_model.pth",
    map_location=DEVICE
)

weighted_model = VehicleCNN(
    num_classes=4
).to(DEVICE)

weighted_model.load_state_dict(
    checkpoint["model_state_dict"]
)

weighted_model.eval()

print("Loaded weighted model")
print("Best epoch:", checkpoint["epoch"])
print("Best F1:", checkpoint["val_f1"])

Loaded weighted model
Best epoch: 9
Best F1: 0.6163510088204303


In [33]:
all_weighted_labels = []
all_weighted_probabilities = []

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = weighted_model(images)

        probabilities = torch.sigmoid(logits)

        all_weighted_labels.append(
            labels.numpy()
        )

        all_weighted_probabilities.append(
            probabilities.cpu().numpy()
        )

all_weighted_labels = np.concatenate(
    all_weighted_labels,
    axis=0
)

all_weighted_probabilities = np.concatenate(
    all_weighted_probabilities,
    axis=0
)

print(
    "Labels:",
    all_weighted_labels.shape
)

print(
    "Probabilities:",
    all_weighted_probabilities.shape
)

Labels: (3798, 4)
Probabilities: (3798, 4)


In [34]:
thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

weighted_best_thresholds = {}
weighted_best_class_f1 = {}

for i, class_name in enumerate(CLASS_NAMES):

    y_true = all_weighted_labels[:, i]
    y_prob = all_weighted_probabilities[:, i]

    best_f1 = 0.0
    best_threshold = 0.5

    for threshold in thresholds:

        predictions = (
            y_prob >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    weighted_best_thresholds[class_name] = best_threshold
    weighted_best_class_f1[class_name] = best_f1

    print(
        f"{class_name:12} | "
        f"Threshold: {best_threshold:.2f} | "
        f"F1: {best_f1:.4f}"
    )

bus          | Threshold: 0.50 | F1: 0.5616
car          | Threshold: 0.40 | F1: 0.8158
motorcycle   | Threshold: 0.50 | F1: 0.5899
truck        | Threshold: 0.45 | F1: 0.5300


In [35]:
weighted_optimized_predictions = np.zeros_like(
    all_weighted_labels
)

for i, class_name in enumerate(CLASS_NAMES):

    weighted_optimized_predictions[:, i] = (
        all_weighted_probabilities[:, i]
        >= weighted_best_thresholds[class_name]
    )

weighted_optimized_f1 = f1_score(
    all_weighted_labels,
    weighted_optimized_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("WEIGHTED MODEL + THRESHOLD OPTIMIZATION")
print("=" * 60)

print(
    f"Weighted F1 @ 0.50 : "
    f"{checkpoint['val_f1']:.4f}"
)

print(
    f"Optimized Macro F1 : "
    f"{weighted_optimized_f1:.4f}"
)

print(
    f"Improvement        : "
    f"{weighted_optimized_f1 - checkpoint['val_f1']:+.4f}"
)


WEIGHTED MODEL + THRESHOLD OPTIMIZATION
Weighted F1 @ 0.50 : 0.6164
Optimized Macro F1 : 0.6243
Improvement        : +0.0080


In [36]:
import json

THRESHOLD_PATH = (
    "/content/AI_TRAFFIC_POLICE/"
    "outputs/training/optimized_thresholds.json"
)

with open(THRESHOLD_PATH, "w") as f:
    json.dump(
        weighted_best_thresholds,
        f,
        indent=4
    )

print("Saved thresholds:")
print(weighted_best_thresholds)
print("\nFile:", THRESHOLD_PATH)

Saved thresholds:
{'bus': np.float64(0.5), 'car': np.float64(0.4), 'motorcycle': np.float64(0.5), 'truck': np.float64(0.45)}

File: /content/AI_TRAFFIC_POLICE/outputs/training/optimized_thresholds.json


In [37]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

import json
import numpy as np
import torch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

# ------------------------------------------------------------
# Load best weighted model
# ------------------------------------------------------------

BEST_WEIGHTED_MODEL = (
    "/content/AI_TRAFFIC_POLICE/"
    "outputs/training/best_weighted_model.pth"
)

checkpoint = torch.load(
    BEST_WEIGHTED_MODEL,
    map_location=DEVICE
)

final_model = VehicleCNN(
    num_classes=4
).to(DEVICE)

final_model.load_state_dict(
    checkpoint["model_state_dict"]
)

final_model.eval()

# ------------------------------------------------------------
# Load validation-selected thresholds
# ------------------------------------------------------------

THRESHOLD_PATH = (
    "/content/AI_TRAFFIC_POLICE/"
    "outputs/training/optimized_thresholds.json"
)

with open(THRESHOLD_PATH, "r") as f:
    threshold_dict = json.load(f)

final_thresholds = np.array([
    threshold_dict["bus"],
    threshold_dict["car"],
    threshold_dict["motorcycle"],
    threshold_dict["truck"],
])

print("Best epoch:", checkpoint["epoch"])
print("Validation F1:", checkpoint["val_f1"])
print("\nFinal thresholds:")

for name, threshold in zip(
    CLASS_NAMES,
    final_thresholds
):
    print(
        f"  {name:12}: {threshold:.2f}"
    )

# ------------------------------------------------------------
# Collect TEST predictions
# ------------------------------------------------------------

test_labels = []
test_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = final_model(images)

        probabilities = torch.sigmoid(
            logits
        )

        test_labels.append(
            labels.numpy()
        )

        test_probabilities.append(
            probabilities.cpu().numpy()
        )

test_labels = np.concatenate(
    test_labels,
    axis=0
)

test_probabilities = np.concatenate(
    test_probabilities,
    axis=0
)

# ------------------------------------------------------------
# Apply per-class thresholds
# ------------------------------------------------------------

test_predictions = np.zeros_like(
    test_labels
)

for i in range(len(CLASS_NAMES)):

    test_predictions[:, i] = (
        test_probabilities[:, i]
        >= final_thresholds[i]
    )

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

test_precision = precision_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

# ------------------------------------------------------------
# Per-class metrics
# ------------------------------------------------------------

class_precision = precision_score(
    test_labels,
    test_predictions,
    average=None,
    zero_division=0
)

class_recall = recall_score(
    test_labels,
    test_predictions,
    average=None,
    zero_division=0
)

class_f1 = f1_score(
    test_labels,
    test_predictions,
    average=None,
    zero_division=0
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(
    f"\nMacro Precision : "
    f"{test_precision:.4f}"
)

print(
    f"Macro Recall    : "
    f"{test_recall:.4f}"
)

print(
    f"Macro F1        : "
    f"{test_f1:.4f}"
)

print("\nPer-class Results")
print("-" * 70)

for i, name in enumerate(CLASS_NAMES):

    print(
        f"{name:12} | "
        f"Precision: {class_precision[i]:.4f} | "
        f"Recall: {class_recall[i]:.4f} | "
        f"F1: {class_f1[i]:.4f}"
    )

Best epoch: 9
Validation F1: 0.6163510088204303

Final thresholds:
  bus         : 0.50
  car         : 0.40
  motorcycle  : 0.50
  truck       : 0.45

FINAL TEST RESULTS

Macro Precision : 0.5533
Macro Recall    : 0.7307
Macro F1        : 0.6180

Per-class Results
----------------------------------------------------------------------
bus          | Precision: 0.5693 | Recall: 0.5583 | F1: 0.5637
car          | Precision: 0.6763 | Recall: 0.9745 | F1: 0.7984
motorcycle   | Precision: 0.5640 | Recall: 0.5782 | F1: 0.5710
truck        | Precision: 0.4034 | Recall: 0.8117 | F1: 0.5390
